# PK/PD Take-Home Exercises

> **Before running anything, read this.**
>
> 1. **R**: Download from https://cran.r-project.org (version 4.2+)
> 2. **R kernel for Jupyter**: In an R console run:
>    ```r
>    install.packages("IRkernel")
>    IRkernel::installspec()
>    ```
>    Then restart Jupyter and select R as the kernel.
> 3. **Packages**: Run the Setup cell below. It tries `renv` first and falls back to `install.packages()` automatically.
>
> Questions? Send them over.

In [ ]:
# Package setup — run this first
setup_ok <- tryCatch({
  if (!requireNamespace("renv", quietly = TRUE))
    install.packages("renv", repos = "https://cloud.r-project.org")
  renv::restore(prompt = FALSE)
  TRUE
}, error = function(e) {
  message("renv failed, using install.packages() instead...")
  FALSE
})

if (!setup_ok) {
  needed  <- c("ggplot2", "dplyr", "nlme", "deSolve", "gridExtra")
  missing <- needed[!sapply(needed, requireNamespace, quietly = TRUE)]
  if (length(missing) > 0)
    install.packages(missing, repos = "https://cloud.r-project.org")
}

library(ggplot2)
library(dplyr)
options(repr.plot.width = 9, repr.plot.height = 4)
cat("Ready.\n")

In [ ]:
# Helper functions — do not modify

pk_oral <- function(t, dose, ka, CL, V, F = 1) {
  ke <- CL / V
  if (abs(ka - ke) < 1e-6) ka <- ka * 1.0001
  (F * dose * ka) / (V * (ka - ke)) * (exp(-ke * t) - exp(-ka * t))
}

one_comp_oral <- function(Time, Dose, ka, CL, V) {
  ke <- CL / V
  (Dose * ka) / (V * (ka - ke)) * (exp(-ke * Time) - exp(-ka * Time))
}

pk_multidose <- function(t_vec, dose, ka, CL, V, interval, n_doses) {
  sapply(t_vec, function(t) {
    dose_times <- seq(0, (n_doses - 1) * interval, by = interval)
    total <- 0
    for (t0 in dose_times)
      if (t >= t0) total <- total + pk_oral(t - t0, dose, ka, CL, V)
    total
  })
}

emax_model <- function(C, Emax, EC50) Emax * C / (EC50 + C)

data(Theoph)
cat("Functions and data loaded.\n")

**Subject 1 parameters from the lecture (for reference):**

| Parameter | Value | Meaning |
|---|---|---|
| ka | 1.777 h⁻¹ | Absorption rate |
| CL | 0.0199 L/kg/h | Clearance |
| V | 0.369 L/kg | Volume of distribution |
| t½ | 12.8 h | Half-life |
| Tmax | ~2.0 h | Time to peak |

## Exercise 1: Fitting the PK Model to Subject 5

In the lecture we fitted Subject 1. Now fit the same model to **Subject 5**, who has noticeably different PK. The code structure is identical — you just need to fill in the subject number and starting value.

In [ ]:
# Step 1: Extract and plot Subject 5's data
sub5 <- subset(Theoph, Subject == ___)   # fill in the subject number

ggplot(sub5, aes(x = Time, y = conc)) +
  geom_point(size = 3, color = "steelblue") +
  geom_line(alpha = 0.3) +
  labs(title = "Subject 5: Concentration–Time Profile",
       x = "Time (h)", y = "Concentration (mg/L)") +
  theme_minimal()

In [ ]:
# Step 2: Fit the oral PK model
# nls() finds the ka, CL, V values that best match the data
fit5 <- nls(
  conc ~ one_comp_oral(Time, Dose, ka, CL, V),
  data  = sub5,
  start = c(ka = ___, CL = 0.04, V = 0.5)   # fill in ka starting value (try 1.5)
)

summary(fit5)

In [ ]:
# Step 3: Extract parameters and compare to Subject 1
p5    <- coef(fit5)
ke5   <- p5[["CL"]] / p5[["V"]]
half5 <- log(2) / ke5

cat("=== Subject 1 vs Subject 5 ===\n")
cat(sprintf("          Subj 1    Subj 5\n"))
cat(sprintf("  ka      1.777     %.3f   h-1\n",     p5[["ka"]]))
cat(sprintf("  CL      0.0199    %.4f  L/kg/h\n",   p5[["CL"]]))
cat(sprintf("  V       0.369     %.3f   L/kg\n",    p5[["V"]]))
cat(sprintf("  t1/2    12.8      %.1f    h\n",      half5))

In [ ]:
# Step 4: Plot observed vs predicted
t_fine <- seq(0, 25, by = 0.1)
pred5  <- data.frame(
  Time = t_fine,
  conc = pk_oral(t_fine, dose = sub5$Dose[1],
                 ka = p5[["ka"]], CL = p5[["CL"]], V = p5[["V"]])
)

ggplot() +
  geom_point(data = sub5,  aes(x = Time, y = conc), size = 3, color = "steelblue") +
  geom_line( data = pred5, aes(x = Time, y = conc), color = "firebrick", linewidth = 1) +
  labs(title = "Subject 5: Observed (points) vs Model (line)",
       x = "Time (h)", y = "Concentration (mg/L)") +
  theme_minimal()

**Question 1a:** Compare Subject 5’s CL and half-life to Subject 1’s. Which subject clears theophylline faster? Theophylline is mainly metabolized by the liver enzyme CYP1A2, and cigarette smoking is a known CYP1A2 inducer. Based on the CL difference, which subject do you think is more likely to be a smoker?

*Your answer:*
[Write here]

---

**Question 1b:** Does the model fit look good? Are there any time points where the model noticeably over- or under-predicts? What might cause a systematic deviation?

*Your answer:*
[Write here]

## Exercise 2: Simulating a Dosing Regimen

A fitted PK model lets us ask questions the original study never tested. Subject 5 received a single dose — but in practice theophylline is given repeatedly. The therapeutic window is:

- **MEC** (minimum effective): 5 mg/L
- **MTC** (maximum tolerated): 15 mg/L

The simulation code below is fully written. Run it and use the output to answer the questions.

In [ ]:
# Subject 5's actual dose in mg (per-kg dose × body weight)
dose_mg  <- sub5$Dose[1] * sub5$Wt[1]
CL_total <- p5[["CL"]] * sub5$Wt[1]
V_total  <- p5[["V"]]  * sub5$Wt[1]

cat(sprintf("Dose: %.0f mg  |  Body weight: %.0f kg  |  t1/2: %.1f h\n",
            dose_mg, sub5$Wt[1], half5))

t_md <- seq(0, 72, by = 0.1)

sim_list <- lapply(c(8, 12, 24), function(tau) {
  data.frame(
    Time     = t_md,
    conc     = pk_multidose(t_md, dose = dose_mg,
                            ka = p5[["ka"]], CL = CL_total, V = V_total,
                            interval = tau, n_doses = ceiling(72 / tau) + 1),
    Interval = paste0("q", tau, "h")
  )
})
sim_df <- do.call(rbind, sim_list)

ggplot(sim_df, aes(x = Time, y = conc, color = Interval)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = 5,  linetype = "dashed", color = "darkgreen") +
  geom_hline(yintercept = 15, linetype = "dashed", color = "red") +
  annotate("text", x = 70, y = 5.7,  label = "MEC 5 mg/L",  hjust = 1, size = 3.5) +
  annotate("text", x = 70, y = 15.7, label = "MTC 15 mg/L", hjust = 1, size = 3.5) +
  labs(title = "Multiple-Dose Simulation: Subject 5 (72 h)",
       x = "Time (h)", y = "Concentration (mg/L)", color = "Interval") +
  theme_minimal()

In [ ]:
# Steady-state trough and peak (t > 48 h is well past 4-5 half-lives for Subject 5)
ss <- sim_df %>%
  filter(Time > ___) %>%   # fill in: use 48 to look at the steady-state window
  group_by(Interval) %>%
  summarise(peak = round(max(conc), 1), trough = round(min(conc), 1), .groups = "drop")

print(ss)

**Question 2a:** Based on the table and plot, which dosing interval (q8h, q12h, or q24h) keeps concentrations within the 5–15 mg/L therapeutic window at steady state? Quote the trough and peak values from your table.

*Your answer:*
[Write here]

---

**Question 2b:** Subject 1 has a half-life of 12.8 h — about twice as long as Subject 5. If you ran the same simulation for Subject 1 at the same dose, would steady state be reached sooner or later? Would you expect the peak concentration at steady state to be higher or lower? Explain your reasoning.

*Your answer:*
[Write here]

## Exercise 3: The Emax Model — Concentration and Effect

So far we have only looked at PK (concentration over time). Now we connect concentration to effect using the Emax model:

$$E(C) = \frac{E_{\max} \cdot C}{EC_{50} + C}$$

- **Emax**: maximum possible effect
- **EC50**: concentration producing half-maximal effect (a measure of potency)

The code below generates synthetic PD data for Subject 4 (with realistic noise), fits the Emax model, and plots the result.

In [ ]:
set.seed(7)
sub4 <- subset(Theoph, Subject == 4)

# True parameters (unknown to the model — it will have to recover them)
Emax_true <- 80   # partial agonist
EC50_true <- 3    # mg/L

sub4$effect <- emax_model(sub4$conc, Emax_true, EC50_true) +
               rnorm(nrow(sub4), 0, sd = 5)
sub4$effect <- pmax(sub4$effect, 0)

ggplot(sub4, aes(x = conc, y = effect)) +
  geom_point(size = 3, color = "steelblue") +
  labs(title = "Subject 4: Concentration–Effect Data",
       x = "Concentration (mg/L)", y = "Effect (%)") +
  theme_minimal()

In [ ]:
# Fit the Emax model
# The formula: predicted effect = Emax * conc / (EC50 + conc)
fit_pd <- nls(
  effect ~ Emax * conc / (EC50 + conc),
  data  = sub4,
  start = list(Emax = ___, EC50 = ___)   # try Emax ~ 70, EC50 ~ 5
)

pd <- coef(fit_pd)
cat(sprintf("Fitted Emax = %.1f%%  (true = %d%%)\n", pd["Emax"], Emax_true))
cat(sprintf("Fitted EC50 = %.2f mg/L  (true = %.0f mg/L)\n", pd["EC50"], EC50_true))

In [ ]:
# Plot the fitted curve over the data
C_seq     <- seq(0, max(sub4$conc) * 1.2, by = 0.05)
fit_curve <- data.frame(conc   = C_seq,
                        effect = emax_model(C_seq, pd[["Emax"]], pd[["EC50"]]))

ggplot() +
  geom_point(data = sub4,      aes(x = conc, y = effect), size = 3, color = "steelblue") +
  geom_line( data = fit_curve, aes(x = conc, y = effect), color = "firebrick", linewidth = 1) +
  geom_hline(yintercept = pd[["Emax"]], linetype = "dotted", color = "gray40") +
  annotate("text", x = 1, y = pd[["Emax"]] + 3,
           label = paste0("Emax = ", round(pd[["Emax"]], 0), "%"), hjust = 0, size = 3.5) +
  labs(title = "Emax Model Fit: Subject 4",
       x = "Concentration (mg/L)", y = "Effect (%)") +
  theme_minimal()

In [ ]:
# Compare this drug (partial agonist) to a full agonist
C_seq2 <- seq(0, 20, by = 0.1)

compare <- data.frame(
  C    = rep(C_seq2, 2),
  Drug = rep(c("Drug X  (Emax=80, EC50=3)  — partial agonist",
               "Drug Y  (Emax=100, EC50=10) — full agonist"), each = length(C_seq2)),
  Emax = rep(c(80, 100), each = length(C_seq2)),
  EC50 = rep(c(3, 10),   each = length(C_seq2))
) %>% mutate(Effect = emax_model(C, Emax, EC50))

ggplot(compare, aes(x = C, y = Effect, color = Drug)) +
  geom_line(linewidth = 1) +
  labs(title = "Potency vs Efficacy: Drug X vs Drug Y",
       x = "Concentration (mg/L)", y = "Effect (%)", color = NULL) +
  theme_minimal() +
  theme(legend.position = "bottom")

**Question 3a:** The fitted Emax is around 80%, not 100%. What does this mean? Can this drug ever achieve full effect no matter how high the dose goes? What is the pharmacological term for a drug with Emax < 100%?

*Your answer:*
[Write here]

---

**Question 3b:** Look at the comparison plot. At low concentrations (say 2 mg/L), which drug produces a stronger effect? At very high concentrations (say 18 mg/L), which drug produces a stronger effect? What property of each drug explains the crossover?

*Your answer:*
[Write here]

---
## Wrapping Up

Bring your completed notebook to next week’s session. We will work through the answers together.

**Think about before class:**
- Subject 5 cleared theophylline ~3× faster than Subject 1. If both received the same fixed dose, which would be at risk of toxicity and which of subtherapeutic levels?
- The Emax curve flattens at high concentrations. What does this mean for the risk-benefit trade-off of very high doses?